In [20]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

## Stage 1: Environment Setup & Data Loading


In [ ]:

from pathlib import Path
import pandas as pd

current_folder = Path.cwd()
print(f"Current working directory: {current_folder}")

project_root = current_folder.parent
print(f"Project root: {project_root}")

raw_folder = project_root / "data" / "raw"

print(f"\nRaw folder: {raw_folder}")
print(f"Raw folder exists: {raw_folder.exists()}")

csv_files = list(raw_folder.glob("*.csv"))

print(f"\nNumber of CSV files found: {len(csv_files)}")
print("CSV files found:")
for file in csv_files:
    print(file)

if len(csv_files) == 1:
    csv_path = csv_files[0]
    print(f"\nSelected CSV: {csv_path}")

    df = pd.read_csv(csv_path)

    print(f"\nShape of raw dataframe: {df.shape}")

    expected_rows = 141712
    expected_cols = 37

    actual_rows, actual_cols = df.shape

    print(f"Rows match expected: {actual_rows == expected_rows}")
    print(f"Columns match expected: {actual_cols == expected_cols}")

    print(f"\nColumn names: \n {df.columns.tolist()}")
    

else:
    raise FileNotFoundError(
        f"Expected exactly 1 CSV file in {raw_folder}, but found {len(csv_files)}. "
        "Check your data/raw folder before continuing."
    )

Current working directory: c:\Users\soumo\agy2-projects\ITSM-Agent-Capstone\notebooks
Project root: c:\Users\soumo\agy2-projects\ITSM-Agent-Capstone

Raw folder: c:\Users\soumo\agy2-projects\ITSM-Agent-Capstone\data\raw
Raw folder exists: True

Number of CSV files found: 1
CSV files found:
c:\Users\soumo\agy2-projects\ITSM-Agent-Capstone\data\raw\incident_event_log_dataset.csv

Selected CSV: c:\Users\soumo\agy2-projects\ITSM-Agent-Capstone\data\raw\incident_event_log_dataset.csv

Shape of raw dataframe: (141712, 37)
Rows match expected: True
Columns match expected: True

Column names: 
 ['number', 'incident_state', 'active', 'reassignment_count', 'reopen_count', 'sys_mod_count', 'made_sla', 'caller_id', 'opened_by', 'opened_at', 'sys_created_by', 'sys_created_at', 'sys_updated_by', 'sys_updated_at', 'contact_type', 'location', 'category', 'subcategory', 'symptom', 'cmdb_ci', 'impact', 'urgency', 'priority', 'assignment_group', 'assigned_to', 'knowledge', 'u_priority_confirmation', 'not

## Stage 2: Working Environment Initialization


In [22]:
df_clean = df.copy()

print(f"Raw dataframe shape : {df.shape}\n")
print(f"Working dataframe(df_clean) shape : {df_clean.shape}\n")

print (f"Is Shape matched for both dataframe? : {df.shape == df_clean.shape} \n Are df and df_clean the same object? : {df is df_clean}")

Raw dataframe shape : (141712, 37)

Working dataframe(df_clean) shape : (141712, 37)

Is Shape matched for both dataframe? : True 
 Are df and df_clean the same object? : False


## Stage 3: Column Validation & MVP Sanity Check


In [23]:
column_names = df_clean.columns.tolist()
print(f"Number of columns : {len(column_names)}\n")
print(f"Column names : {column_names}\n")


columns_with_spaces = [
    col for col in df_clean.columns
    if col != col.strip()
]
if columns_with_spaces:
    print("Decision: Column names have spacing issues. Rename only if justified.")
else:
    print("Decision: No leading/trailing space issues found.")



columns_with_uppercase = [
    col for col in df_clean.columns 
    if col != col.lower()
]
if columns_with_uppercase:
    print(f"\nDecision: Uppercase found in {columns_with_uppercase} , but no rename applied now because the MVP pipeline does not require renaming them."
    )
else:
    print("\nDecision: No uppercase column issues found.")



expected_mvp_cols = [
    "number",
    "incident_state",
    "active",
    "made_sla",
    "opened_at",
    "sys_updated_at",
    "resolved_at",
    "closed_at",
    "priority",
    "impact",
    "urgency",
    "category",
    "subcategory",
    "assignment_group",
    "assigned_to",
    "reassignment_count",
    "reopen_count",
    "sys_mod_count",
    "problem_id",
    "change",
    "caused_by",
    "closed_code"
]
missing_mvp_cols = [
    col for col in expected_mvp_cols
    if col not in df_clean.columns
]
if len(missing_mvp_cols) == 0:
    print("\nDecision: All MVP columns exist.")
else:
    print("\nDecision: Stop and investigate missing MVP columns before continuing.")




Number of columns : 37

Column names : ['number', 'incident_state', 'active', 'reassignment_count', 'reopen_count', 'sys_mod_count', 'made_sla', 'caller_id', 'opened_by', 'opened_at', 'sys_created_by', 'sys_created_at', 'sys_updated_by', 'sys_updated_at', 'contact_type', 'location', 'category', 'subcategory', 'symptom', 'cmdb_ci', 'impact', 'urgency', 'priority', 'assignment_group', 'assigned_to', 'knowledge', 'u_priority_confirmation', 'notify', 'problem_id', 'change', 'vendor', 'caused_by', 'closed_code', 'resolved_by', 'resolved_at', 'closed_at', 'isParent']

Decision: No leading/trailing space issues found.

Decision: Uppercase found in ['isParent'] , but no rename applied now because the MVP pipeline does not require renaming them.

Decision: All MVP columns exist.


## Stage 4: Replace Fake Missing Values

In [24]:


# 1. Define fake missing tokens
fake_missing_values = ["?", "NA", "N/A", "None", "null", "Unknown", "-", ""]

# 2. Choose columns to inspect carefully
fake_missing_cols = ['caller_id', 'opened_by', 'sys_created_by', 'sys_created_at', 'location', 'category', 'subcategory', 'symptom', 'cmdb_ci', 
                    'assignment_group', 'assigned_to', 'problem_id', 'change', 'vendor', 'caused_by', 'closed_code', 'resolved_by', 'resolved_at']

# 3. Count fake missing values BEFORE replacement
fake_missing_before = []

for col in fake_missing_cols:
    fake_count = (
        df_clean[col]
        .astype(str)
        .str.strip()
        .isin(fake_missing_values)
        .sum()
    )

    fake_values_found = (
    df_clean[col]
    .astype(str)
    .str.strip()
    .loc[
        df_clean[col]
        .astype(str)
        .str.strip()
        .isin(fake_missing_values)
    ]
    .value_counts()
    .to_dict()
    )

    real_missing_count = df_clean[col].isna().sum()

    fake_missing_before.append({
        "column": col,
        "real_missing_before": real_missing_count,
        "fake_missing_before": fake_count,
        "fake_values_found": fake_values_found
     })

fake_missing_before_df = pd.DataFrame(fake_missing_before)
fake_missing_before_df 


# 4. Replace fake missing values with proper missing values
for col in fake_missing_cols:
    df_clean[col] = (
        df_clean[col]
        .replace(fake_missing_values, pd.NA)
    )


# 5. Count missing values AFTER replacement
fake_missing_after = []

for col in fake_missing_cols:
    remaining_fake_count = (
        df_clean[col]
        .astype(str)
        .str.strip()
        .isin(fake_missing_values)
        .sum()
    )


    missing_after = df_clean[col].isna().sum()


    fake_missing_after.append({
        "column": col,
        "remaining_fake_values_after": remaining_fake_count,
        "missing_after_replace": missing_after
    })

fake_missing_after_df = pd.DataFrame(fake_missing_after)
fake_missing_after_df

#6 To check if the replacement happened successfully 
df_clean[["number"] + fake_missing_cols].head(10)



,column,real_missing_before,fake_missing_before,fake_values_found
0,caller_id,0,29,{'?': 29}
1,opened_by,0,4835,{'?': 4835}
2,sys_created_by,0,53076,{'?': 53076}
3,sys_created_at,0,53076,{'?': 53076}
4,location,0,76,{'?': 76}
5,category,0,78,{'?': 78}
6,subcategory,0,111,{'?': 111}
7,symptom,0,32964,{'?': 32964}
8,cmdb_ci,0,141267,{'?': 141267}
9,assignment_group,0,14213,{'?': 14213}


,column,remaining_fake_values_after,missing_after_replace
0,caller_id,0,29
1,opened_by,0,4835
2,sys_created_by,0,53076
3,sys_created_at,0,53076
4,location,0,76
5,category,0,78
6,subcategory,0,111
7,symptom,0,32964
8,cmdb_ci,0,141267
9,assignment_group,0,14213


,number,caller_id,opened_by,sys_created_by,sys_created_at,location,category,subcategory,symptom,cmdb_ci,assignment_group,assigned_to,problem_id,change,vendor,caused_by,closed_code,resolved_by,resolved_at
0,INC0000045,Caller 2403,Opened by 8,Created by 6,29-02-2016 01:23,Location 143,Category 55,Subcategory 170,Symptom 72,NaN,Group 56,NaN,NaN,NaN,NaN,NaN,code 5,Resolved by 149,29-02-2016 11:29
1,INC0000045,Caller 2403,Opened by 8,Created by 6,29-02-2016 01:23,Location 143,Category 55,Subcategory 170,Symptom 72,NaN,Group 56,NaN,NaN,NaN,NaN,NaN,code 5,Resolved by 149,29-02-2016 11:29
2,INC0000045,Caller 2403,Opened by 8,Created by 6,29-02-2016 01:23,Location 143,Category 55,Subcategory 170,Symptom 72,NaN,Group 56,NaN,NaN,NaN,NaN,NaN,code 5,Resolved by 149,29-02-2016 11:29
3,INC0000045,Caller 2403,Opened by 8,Created by 6,29-02-2016 01:23,Location 143,Category 55,Subcategory 170,Symptom 72,NaN,Group 56,NaN,NaN,NaN,NaN,NaN,code 5,Resolved by 149,29-02-2016 11:29
4,INC0000047,Caller 2403,Opened by 397,Created by 171,29-02-2016 04:57,Location 165,Category 40,Subcategory 215,Symptom 471,NaN,Group 70,Resolver 89,NaN,NaN,NaN,NaN,code 5,Resolved by 81,01-03-2016 09:52
5,INC0000047,Caller 2403,Opened by 397,Created by 171,29-02-2016 04:57,Location 165,Category 40,Subcategory 215,Symptom 471,NaN,Group 24,Resolver 31,NaN,NaN,NaN,NaN,code 5,Resolved by 81,01-03-2016 09:52
6,INC0000047,Caller 2403,Opened by 397,Created by 171,29-02-2016 04:57,Location 165,Category 40,Subcategory 215,Symptom 471,NaN,Group 24,Resolver 31,NaN,NaN,NaN,NaN,code 5,Resolved by 81,01-03-2016 09:52
7,INC0000047,Caller 2403,Opened by 397,Created by 171,29-02-2016 04:57,Location 165,Category 40,Subcategory 215,Symptom 471,NaN,Group 24,Resolver 31,NaN,NaN,NaN,NaN,code 5,Resolved by 81,01-03-2016 09:52
8,INC0000047,Caller 2403,Opened by 397,Created by 171,29-02-2016 04:57,Location 165,Category 40,Subcategory 215,Symptom 471,NaN,Group 24,Resolver 31,NaN,NaN,NaN,NaN,code 5,Resolved by 81,01-03-2016 09:52
9,INC0000047,Caller 2403,Opened by 397,Created by 171,29-02-2016 04:57,Location 165,Category 40,Subcategory 215,Symptom 471,NaN,Group 24,Resolver 31,NaN,NaN,NaN,NaN,code 5,Resolved by 81,01-03-2016 09:52


## Stage 5: Convert Datetime Columns & Production Check

In [25]:

datetime_cols = [
    "opened_at",
    "sys_created_at",
    "sys_updated_at",
    "resolved_at",
    "closed_at"
]

# 1. Check current data types before conversion
print(f"Data types BEFORE datetime conversion: \n{df_clean[datetime_cols].dtypes}")


# 2. Count missing values before conversion
missing_before_datetime = df_clean[datetime_cols].isna().sum()
print(f"\nMissing values BEFORE datetime conversion: \n{missing_before_datetime}")


# 3. Convert each datetime column
for col in datetime_cols:
    df_clean[col] = pd.to_datetime(
        df_clean[col],
        errors="coerce",
        dayfirst=True
    )

# 4. Check data types after conversion
print(f"\nData types AFTER datetime conversion: \n{df_clean[datetime_cols].dtypes}")


# 5. Count NaT values after conversion
missing_after_datetime = df_clean[datetime_cols].isna().sum()
print(f"\nMissing / NaT values AFTER datetime conversion: \n{missing_after_datetime}")


# 6. Confirm sys_updated_at is usable for latest-row logic
missing_sys_updated_at = df_clean["sys_updated_at"].isna().sum()

print(f"\nMissing sys_updated_at values: {missing_sys_updated_at}")

if missing_sys_updated_at == 0:
    print("Decision: sys_updated_at is usable for latest-row selection.")
else:
    print("Decision: sys_updated_at has missing values. Latest-row logic needs extra caution.")



# 7. Lightweight lifecycle issue counts
opened_after_resolved = (
    df_clean["opened_at"].notna()
    & df_clean["resolved_at"].notna()
    & (df_clean["opened_at"] > df_clean["resolved_at"])
)

resolved_after_closed = (
    df_clean["resolved_at"].notna()
    & df_clean["closed_at"].notna()
    & (df_clean["resolved_at"] > df_clean["closed_at"])
)

print(f"\nRows where opened_at is after resolved_at:{opened_after_resolved.sum()}")
print(f"Rows where resolved_at is after closed_at:{resolved_after_closed.sum()}")


Data types BEFORE datetime conversion: 
opened_at         str
sys_created_at    str
sys_updated_at    str
resolved_at       str
closed_at         str
dtype: object

Missing values BEFORE datetime conversion: 
opened_at             0
sys_created_at    53076
sys_updated_at        0
resolved_at        3141
closed_at             0
dtype: int64

Data types AFTER datetime conversion: 
opened_at         datetime64[us]
sys_created_at    datetime64[us]
sys_updated_at    datetime64[us]
resolved_at       datetime64[us]
closed_at         datetime64[us]
dtype: object

Missing / NaT values AFTER datetime conversion: 
opened_at             0
sys_created_at    53076
sys_updated_at        0
resolved_at        3141
closed_at             0
dtype: int64

Missing sys_updated_at values: 0
Decision: sys_updated_at is usable for latest-row selection.

Rows where opened_at is after resolved_at:0
Rows where resolved_at is after closed_at:0


## Stage 7: Validate Event-Level Grain Again

In [26]:

total_rows = len(df_clean)
unique_incidents = df_clean["number"].nunique()

rows_per_incident = df_clean["number"].value_counts()


duplicate_full_rows = df_clean.duplicated().sum()

print(f"Total event-level rows: {total_rows}")
print(f"Unique incidents: {unique_incidents}")
print(f"Average rows per incident: {total_rows / unique_incidents}")
print(f"Duplicate full rows: {duplicate_full_rows}")

print(f"\nRows per incident summary: \n {rows_per_incident.describe()}")

print(f"\nTop repeated incidents: \n {rows_per_incident.head(10)}")


Total event-level rows: 141712
Unique incidents: 24918
Average rows per incident: 5.687133798860262
Duplicate full rows: 0

Rows per incident summary: 
 count    24918.000000
mean         5.687134
std          3.677845
min          2.000000
25%          3.000000
50%          5.000000
75%          7.000000
max         58.000000
Name: count, dtype: float64

Top repeated incidents: 
 number
INC0019396    58
INC0044260    56
INC0005927    46
INC0020718    45
INC0011206    44
INC0007349    43
INC0025734    43
INC0012815    40
INC0003419    38
INC0020849    38
Name: count, dtype: int64


## Stage 8: Create Latest-Row Snapshot Per Incident

In [27]:

# 1. Sort rows so the latest row per incident appears last
df_sorted = df_clean.sort_values(
    by=["number", "sys_updated_at", "sys_mod_count"],
    ascending=[True, True, True]
)

# 2. Take the last row from each incident group
latest_incident_rows = (
    df_sorted
    .groupby("number")
    .tail(1)
    .copy()
)

# 3. Validate shape
print(f"Raw event-level rows:{len(df_clean)}")
print(f"Unique incidents:{df_clean['number'].nunique()}")
print(f"Latest snapshot rows: {len(latest_incident_rows)}")

# 4. Validate one row per incident
duplicate_incidents_after_snapshot = latest_incident_rows["number"].duplicated().sum()

print(f"Duplicate incident numbers in latest snapshot: {duplicate_incidents_after_snapshot}")

# 5. Preview latest snapshot
latest_incident_rows.head()

print(f"Validation Check: Does every unique ticket have exactly one final row? : {len(latest_incident_rows) == df_clean['number'].nunique()}")


Raw event-level rows:141712
Unique incidents:24918
Latest snapshot rows: 24918
Duplicate incident numbers in latest snapshot: 0


,number,incident_state,active,reassignment_count,reopen_count,sys_mod_count,made_sla,caller_id,opened_by,opened_at,...,notify,problem_id,change,vendor,caused_by,closed_code,resolved_by,resolved_at,closed_at,isParent
3,INC0000045,Closed,False,0,0,4,True,Caller 2403,Opened by 8,2016-02-29 01:16:00,...,Do Not Notify,NaN,NaN,NaN,NaN,code 5,Resolved by 149,2016-02-29 11:29:00,2016-03-05 12:00:00,No
12,INC0000047,Closed,False,1,0,8,True,Caller 2403,Opened by 397,2016-02-29 04:40:00,...,Do Not Notify,NaN,NaN,NaN,NaN,code 5,Resolved by 81,2016-03-01 09:52:00,2016-03-06 10:00:00,No
19,INC0000057,Closed,False,0,0,6,True,Caller 4416,Opened by 8,2016-02-29 06:10:00,...,Do Not Notify,Problem ID 2,NaN,NaN,NaN,code 10,Resolved by 5,2016-03-01 02:55:00,2016-03-06 03:00:00,No
23,INC0000060,Closed,False,0,0,3,True,Caller 4491,Opened by 180,2016-02-29 06:38:00,...,Do Not Notify,NaN,NaN,NaN,NaN,code 3,Resolved by 113,2016-03-02 12:06:00,2016-03-07 13:00:00,No
31,INC0000062,Closed,False,1,0,7,False,Caller 3765,Opened by 180,2016-02-29 06:58:00,...,Do Not Notify,NaN,NaN,NaN,NaN,code 7,Resolved by 62,2016-02-29 15:51:00,2016-03-05 16:00:00,No


Validation Check: Does every unique ticket have exactly one final row? : True


## Stage 9: Aggregate counters per incident.

In [28]:

counter_cols = [
    "reassignment_count",
    "reopen_count",
    "sys_mod_count"
]

# 1. Check counter column data types
print(f"Counter column data types:\n{df_clean[counter_cols].dtypes}\n")

# 2. Create max counter summary per incident
counter_summary = (
    df_clean
    .groupby("number")[counter_cols]
    .agg("max")
    .reset_index()
)

# 3. Validate counter summary
unique_incidents = df_clean["number"].nunique()
summary_rows = len(counter_summary)
duplicates = counter_summary["number"].duplicated().sum()

print(f"Unique incidents in df_clean: {unique_incidents}")
print(f"Rows in counter summary: {summary_rows}")
print(f"Validation Check (Rows == Unique Incidents?): {unique_incidents == summary_rows}")
print(f"Duplicate incident numbers in counter summary: {duplicates}")

# 4. Preview
counter_summary.head()

print(f"Validation Check: Does every unique ticket have aggregated counters? : {len(counter_summary) == df_clean['number'].nunique()}")


Counter column data types:
reassignment_count    int64
reopen_count          int64
sys_mod_count         int64
dtype: object

Unique incidents in df_clean: 24918
Rows in counter summary: 24918
Validation Check (Rows == Unique Incidents?): True
Duplicate incident numbers in counter summary: 0


,number,reassignment_count,reopen_count,sys_mod_count
0,INC0000045,0,0,4
1,INC0000047,1,0,8
2,INC0000057,0,0,6
3,INC0000060,0,0,3
4,INC0000062,1,0,7


Validation Check: Does every unique ticket have aggregated counters? : True


## Stage 10: Build the Incident-Level-Ready Dataset

In [29]:


latest_row_cols = [
    "number",
    "incident_state",
    "active",
    "made_sla",
    "opened_at",
    "sys_updated_at",
    "resolved_at",
    "closed_at",
    "priority",
    "impact",
    "urgency",
    "category",
    "subcategory",
    "assignment_group",
    "assigned_to",
    "problem_id",
    "change",
    "caused_by",
    "closed_code"
]

# 2. Create latest-row base table
latest_base = latest_incident_rows[latest_row_cols].copy()

# 3. Merge latest row fields with aggregated counters
incidents_level_df = latest_base.merge(
    counter_summary,                                    #--------------> from stage 9
    on="number",
    how="left"
)

# 4. Validate final shape
print(f"Rows in latest_base: {len(latest_base)}")
print(f"Rows in counter_summary: {len(counter_summary)}")
print(f"Rows in incidents_level_df: {len(incidents_level_df)}")

# 5. Validate one row per incident
print(f"Duplicate incident numbers: {incidents_level_df['number'].duplicated().sum()}")

print(
    f"Validation Check: Is Incident-Level dataset one row per incident?: {len(incidents_level_df) == incidents_level_df['number'].nunique()}"
)

# 6. Preview
incidents_level_df.head()

Rows in latest_base: 24918
Rows in counter_summary: 24918
Rows in incidents_level_df: 24918
Duplicate incident numbers: 0
Validation Check: Is Incident-Level dataset one row per incident?: True


,number,incident_state,active,made_sla,opened_at,sys_updated_at,resolved_at,closed_at,priority,impact,...,subcategory,assignment_group,assigned_to,problem_id,change,caused_by,closed_code,reassignment_count,reopen_count,sys_mod_count
0,INC0000045,Closed,False,True,2016-02-29 01:16:00,2016-03-05 12:00:00,2016-02-29 11:29:00,2016-03-05 12:00:00,3 - Moderate,2 - Medium,...,Subcategory 170,Group 56,NaN,NaN,NaN,NaN,code 5,0,0,4
1,INC0000047,Closed,False,True,2016-02-29 04:40:00,2016-03-06 10:00:00,2016-03-01 09:52:00,2016-03-06 10:00:00,3 - Moderate,2 - Medium,...,Subcategory 215,Group 24,Resolver 89,NaN,NaN,NaN,code 5,1,0,8
2,INC0000057,Closed,False,True,2016-02-29 06:10:00,2016-03-06 03:00:00,2016-03-01 02:55:00,2016-03-06 03:00:00,3 - Moderate,2 - Medium,...,Subcategory 125,Group 70,Resolver 6,Problem ID 2,NaN,NaN,code 10,0,0,6
3,INC0000060,Closed,False,True,2016-02-29 06:38:00,2016-03-07 13:00:00,2016-03-02 12:06:00,2016-03-07 13:00:00,3 - Moderate,2 - Medium,...,Subcategory 97,Group 25,Resolver 125,NaN,NaN,NaN,code 3,0,0,3
4,INC0000062,Closed,False,False,2016-02-29 06:58:00,2016-03-05 16:00:00,2016-02-29 15:51:00,2016-03-05 16:00:00,2 - High,1 - High,...,Subcategory 168,Group 23,NaN,NaN,NaN,NaN,code 7,1,0,7


## Stage 11: Adding Derived Fields for new incident level dataframe


In [40]:

# 1. Create resolution_hours only when resolved_at and opened_at exist
incidents_level_df["resolution_hours"] = (
    incidents_level_df["resolved_at"] - incidents_level_df["opened_at"]
).dt.total_seconds() / 3600

# 2. Create SLA breach flag
incidents_level_df["is_sla_breached"] = (
    incidents_level_df["made_sla"] == False
)

# 3. Create link flags
incidents_level_df["has_problem_link"] = incidents_level_df["problem_id"].notna()
incidents_level_df["has_change_link"] = incidents_level_df["change"].notna()
incidents_level_df["has_caused_by_link"] = incidents_level_df["caused_by"].notna()

# 4. Create counter-based flags
incidents_level_df["is_reopened"] = incidents_level_df["reopen_count"] > 0
incidents_level_df["was_reassigned"] = incidents_level_df["reassignment_count"] > 0

# 5. Preview new columns
derived_cols = [
    "number",
    "resolution_hours",
    "is_sla_breached",
    "has_problem_link",
    "has_change_link",
    "has_caused_by_link",
    "is_reopened",
    "was_reassigned"
]

incidents_level_df.head()

# Validation :-
print(f"Resolution hours summary: \n{incidents_level_df['resolution_hours'].describe()}")

print(f"\nMissing resolution_hours: {incidents_level_df['resolution_hours'].isna().sum()}")

print(f"\nNegative resolution_hours: {(incidents_level_df['resolution_hours'] < 0).sum()}")



flag_cols = [
    "is_sla_breached",
    "has_problem_link",
    "has_change_link",
    "has_caused_by_link",
    "is_reopened",
    "was_reassigned"
]
for col in flag_cols:
    
    print(f"\n{incidents_level_df[col].value_counts(dropna=False)}")

,number,incident_state,active,made_sla,opened_at,sys_updated_at,resolved_at,closed_at,priority,impact,...,reassignment_count,reopen_count,sys_mod_count,resolution_hours,is_sla_breached,has_problem_link,has_change_link,has_caused_by_link,is_reopened,was_reassigned
0,INC0000045,Closed,False,True,2016-02-29 01:16:00,2016-03-05 12:00:00,2016-02-29 11:29:00,2016-03-05 12:00:00,3 - Moderate,2 - Medium,...,0,0,4,10.216667,False,False,False,False,False,False
1,INC0000047,Closed,False,True,2016-02-29 04:40:00,2016-03-06 10:00:00,2016-03-01 09:52:00,2016-03-06 10:00:00,3 - Moderate,2 - Medium,...,1,0,8,29.200000,False,False,False,False,False,True
2,INC0000057,Closed,False,True,2016-02-29 06:10:00,2016-03-06 03:00:00,2016-03-01 02:55:00,2016-03-06 03:00:00,3 - Moderate,2 - Medium,...,0,0,6,20.750000,False,True,False,False,False,False
3,INC0000060,Closed,False,True,2016-02-29 06:38:00,2016-03-07 13:00:00,2016-03-02 12:06:00,2016-03-07 13:00:00,3 - Moderate,2 - Medium,...,0,0,3,53.466667,False,False,False,False,False,False
4,INC0000062,Closed,False,False,2016-02-29 06:58:00,2016-03-05 16:00:00,2016-02-29 15:51:00,2016-03-05 16:00:00,2 - High,1 - High,...,1,0,7,8.883333,True,False,False,False,False,True


Resolution hours summary: 
count    23362.000000
mean       178.171582
std        532.787772
min          0.000000
25%          0.416667
50%         22.100000
75%        148.479167
max       8070.166667
Name: resolution_hours, dtype: float64

Missing resolution_hours: 1556

Negative resolution_hours: 0

is_sla_breached
False    15803
True      9115
Name: count, dtype: int64

has_problem_link
False    24538
True       380
Name: count, dtype: int64

has_change_link
False    24742
True       176
Name: count, dtype: int64

has_caused_by_link
False    24915
True         3
Name: count, dtype: int64

is_reopened
False    24643
True       275
Name: count, dtype: int64

was_reassigned
False    13549
True     11369
Name: count, dtype: int64


## Stage 12: Final Validation Before Export


In [44]:

# 1. Basic row-level validation
total_rows = len(incidents_level_df)
unique_incidents = incidents_level_df["number"].nunique()
duplicate_incidents = incidents_level_df["number"].duplicated().sum()

print(f"Total rows:{total_rows}")
print(f"Unique incidents:{unique_incidents}")
print(f"Duplicate incident numbers: {duplicate_incidents}")
print(f"Final shape: {incidents_level_df.shape}")

print(f"\nFinal data types: {incidents_level_df.info()}")



print(
    f"Validation Check: One row per incident: {total_rows == unique_incidents and duplicate_incidents == 0}"
    )

# 2. Required column validation
required_cols = [
    "number",
    "incident_state",
    "active",
    "made_sla",
    "opened_at",
    "sys_updated_at",
    "resolved_at",
    "closed_at",
    "priority",
    "impact",
    "urgency",
    "category",
    "subcategory",
    "assignment_group",
    "assigned_to",
    "reassignment_count",
    "reopen_count",
    "sys_mod_count",
    "problem_id",
    "change",
    "caused_by",
    "closed_code",
    "resolution_hours",
    "is_sla_breached",
    "has_problem_link",
    "has_change_link",
    "has_caused_by_link",
    "is_reopened",
    "was_reassigned"
]

missing_required_cols = [
    col for col in required_cols
    if col not in incidents_level_df.columns
]

print(f"Missing required columns: {missing_required_cols}")


missing_summary = incidents_level_df.isna().sum()
missing_summary

Total rows:24918
Unique incidents:24918
Duplicate incident numbers: 0
Final shape: (24918, 29)
<class 'pandas.DataFrame'>
RangeIndex: 24918 entries, 0 to 24917
Data columns (total 29 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   number              24918 non-null  str           
 1   incident_state      24918 non-null  str           
 2   active              24918 non-null  bool          
 3   made_sla            24918 non-null  bool          
 4   opened_at           24918 non-null  datetime64[us]
 5   sys_updated_at      24918 non-null  datetime64[us]
 6   resolved_at         23362 non-null  datetime64[us]
 7   closed_at           24918 non-null  datetime64[us]
 8   priority            24918 non-null  str           
 9   impact              24918 non-null  str           
 10  urgency             24918 non-null  str           
 11  category            24911 non-null  str           
 12  subcategory       

number                    0
incident_state            0
active                    0
made_sla                  0
opened_at                 0
sys_updated_at            0
resolved_at            1556
closed_at                 0
priority                  0
impact                    0
urgency                   0
category                  7
subcategory               8
assignment_group       2157
assigned_to             725
problem_id            24538
change                24742
caused_by             24915
closed_code             107
reassignment_count        0
reopen_count              0
sys_mod_count             0
resolution_hours       1556
is_sla_breached           0
has_problem_link          0
has_change_link           0
has_caused_by_link        0
is_reopened               0
was_reassigned            0
dtype: int64

## Stage 13: Export Processed Dataset


In [ ]:

from pathlib import Path

# 1. Define processed folder
processed_folder = project_root / "data" / "processed"

# 2. Create processed folder if it does not exist
processed_folder.mkdir(parents=True, exist_ok=True)

# 3. Define output file path
output_path = processed_folder / "incidents_level_df.csv"

print(f"Output path: {output_path}")

# 4. Export incidents_level_df to CSV
incidents_level_df.to_csv(output_path, index=False)

# 5. Confirm file exists
print(f"File exported successfully: {output_path.exists()}")

export_check_df = pd.read_csv(output_path)

print(f"Original shape: {incidents_level_df.shape}")
print(f"Reloaded shape: {export_check_df.shape}")
print(f"Shape match: {incidents_level_df.shape == export_check_df.shape}")

Output path: c:\Users\soumo\agy2-projects\ITSM-Agent-Capstone\data\processed\incidents_level_df.csv
File exported successfully: True
Original shape: (24918, 29)
Reloaded shape: (24918, 29)
Shape match: True
